In [9]:
%pip install --upgrade pip
%pip install --upgrade --no-cache-dir --only-binary=:all: "transformers==4.57.6" "tokenizers==0.22.2" "jedi>=0.16" "setuptools<82"

In [10]:
import re
import random
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertConfig, BertForSequenceClassification

random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(2)
print("Device:", device)

Device: cuda


In [11]:
positive = [
    "I love this movie", "This movie is wonderful", "The film was excellent",
    "I enjoyed the story", "The acting was brilliant", "This is a great film",
    "The ending made me happy", "I really liked the characters",
    "The music was beautiful", "This was an amazing experience",
    "I love this product", "The product works perfectly", "The quality is excellent",
    "I am happy with my purchase", "This phone is fantastic", "The battery is great",
    "This is worth the money", "I highly recommend this product",
    "The design is beautiful", "This item is very useful",
    "The service was excellent", "The staff were friendly", "I enjoyed my visit",
    "The food was delicious", "The restaurant is wonderful", "The meal was amazing",
    "The delivery was quick", "The team was helpful", "I had a great day",
    "Everything was perfect", "I feel happy today", "This made me smile",
    "I am pleased with the result", "The experience was pleasant",
    "The app is easy to use", "This book is interesting", "I loved the performance",
    "The support was fantastic", "I am satisfied", "This is good"
]
negative = [
    "I hate this movie", "This movie is terrible", "The film was awful",
    "I disliked the story", "The acting was poor", "This is a bad film",
    "The ending made me sad", "I really disliked the characters",
    "The music was annoying", "This was a horrible experience",
    "I hate this product", "The product does not work", "The quality is terrible",
    "I am unhappy with my purchase", "This phone is disappointing", "The battery is bad",
    "This is a waste of money", "I cannot recommend this product",
    "The design is ugly", "This item is useless",
    "The service was awful", "The staff were rude", "I disliked my visit",
    "The food was disgusting", "The restaurant is terrible", "The meal was horrible",
    "The delivery was slow", "The team was unhelpful", "I had a bad day",
    "Everything was disappointing", "I feel sad today", "This made me angry",
    "I am upset with the result", "The experience was unpleasant",
    "The app is difficult to use", "This book is boring", "I hated the performance",
    "The support was poor", "I am dissatisfied", "This is bad"
]
train_data = [(s, 1) for s in positive] + [(s, 0) for s in negative]
random.shuffle(train_data)
test_data = [
    ("The movie was great", 1), ("I enjoyed this film", 1),
    ("The food was excellent", 1), ("The service was wonderful", 1),
    ("I am happy with this product", 1), ("The staff were helpful", 1),
    ("This app is fantastic", 1), ("The book was interesting", 1),
    ("I loved this experience", 1), ("The quality was good", 1),
    ("The movie was bad", 0), ("I hated this film", 0),
    ("The food was terrible", 0), ("The service was horrible", 0),
    ("I am unhappy with this product", 0), ("The staff were unhelpful", 0),
    ("This app is disappointing", 0), ("The book was boring", 0),
    ("I disliked this experience", 0), ("The quality was poor", 0)
]
assert not set(s for s, _ in train_data) & set(s for s, _ in test_data)
print("Training examples:", len(train_data))
print("Test examples:", len(test_data))

Training examples: 80
Test examples: 20


In [12]:
def tokenize(text):
    return re.findall(r"[a-z]+|[0-9]+|[^\w\s]", text.lower())

special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
words = sorted({word for text, _ in train_data for word in tokenize(text)})
vocab = {word: i for i, word in enumerate(special_tokens + words)}
MAX_LENGTH = 32

def encode(text):
    tokens = ["[CLS]"] + tokenize(text)[:MAX_LENGTH - 2] + ["[SEP]"]
    ids = [vocab.get(token, vocab["[UNK]"]) for token in tokens]
    mask = [1] * len(ids)
    padding = MAX_LENGTH - len(ids)
    return ids + [vocab["[PAD]"]] * padding, mask + [0] * padding

def make_dataset(data):
    encoded = [encode(text) for text, _ in data]
    return TensorDataset(
        torch.tensor([ids for ids, _ in encoded], dtype=torch.long),
        torch.tensor([mask for _, mask in encoded], dtype=torch.long),
        torch.tensor([label for _, label in data], dtype=torch.long)
    )

train_loader = DataLoader(make_dataset(train_data), batch_size=16, shuffle=True)
test_loader = DataLoader(make_dataset(test_data), batch_size=16)
print("Vocabulary size:", len(vocab))
print("Example tokens:", tokenize("I love this movie"))

Vocabulary size: 116
Example tokens: ['i', 'love', 'this', 'movie']


In [13]:
config = BertConfig(
    vocab_size=len(vocab),
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=4,
    intermediate_size=128,
    max_position_embeddings=MAX_LENGTH,
    pad_token_id=vocab["[PAD]"],
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)
model = BertForSequenceClassification(config).to(device)
print("Randomly initialized BERT created.")
print("Parameters:", sum(p.numel() for p in model.parameters()))

Randomly initialized BERT created.
Parameters: 80962


In [14]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    for ids, mask, labels in train_loader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.logits.argmax(dim=1) == labels).sum().item()
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Loss: {total_loss / len(train_data):.4f} | "
          f"Training accuracy: {correct / len(train_data):.1%}")

Epoch 01/30 | Loss: 0.6970 | Training accuracy: 48.8%
Epoch 02/30 | Loss: 0.6937 | Training accuracy: 48.8%
Epoch 03/30 | Loss: 0.6947 | Training accuracy: 51.2%
Epoch 04/30 | Loss: 0.6965 | Training accuracy: 50.0%
Epoch 05/30 | Loss: 0.6923 | Training accuracy: 53.8%
Epoch 06/30 | Loss: 0.6905 | Training accuracy: 62.5%
Epoch 07/30 | Loss: 0.6743 | Training accuracy: 73.8%
Epoch 08/30 | Loss: 0.5814 | Training accuracy: 91.2%
Epoch 09/30 | Loss: 0.3302 | Training accuracy: 100.0%
Epoch 10/30 | Loss: 0.1254 | Training accuracy: 100.0%
Epoch 11/30 | Loss: 0.0465 | Training accuracy: 100.0%
Epoch 12/30 | Loss: 0.0203 | Training accuracy: 100.0%
Epoch 13/30 | Loss: 0.0104 | Training accuracy: 100.0%
Epoch 14/30 | Loss: 0.0068 | Training accuracy: 100.0%
Epoch 15/30 | Loss: 0.0046 | Training accuracy: 100.0%
Epoch 16/30 | Loss: 0.0038 | Training accuracy: 100.0%
Epoch 17/30 | Loss: 0.0031 | Training accuracy: 100.0%
Epoch 18/30 | Loss: 0.0026 | Training accuracy: 100.0%
Epoch 19/30 | Loss

In [15]:
model.eval()
correct = 0
with torch.no_grad():
    for ids, mask, labels in test_loader:
        outputs = model(input_ids=ids.to(device), attention_mask=mask.to(device))
        predictions = outputs.logits.argmax(dim=1).cpu()
        correct += (predictions == labels).sum().item()
print(f"Test accuracy: {correct}/{len(test_data)} = {correct / len(test_data):.1%}")

Test accuracy: 20/20 = 100.0%


In [16]:
text = "I really liked the characters and the story."
if not text:
    print("Please enter a non-empty sentence and run this cell again.")
else:
    ids, mask = encode(text)
    model.eval()
    with torch.no_grad():
        outputs = model(
            input_ids=torch.tensor([ids], dtype=torch.long, device=device),
            attention_mask=torch.tensor([mask], dtype=torch.long, device=device)
        )
    probabilities = torch.softmax(outputs.logits, dim=1)
    prediction = torch.argmax(probabilities, dim=1).item()
    labels = ["NEGATIVE", "POSITIVE"]
    print("Sentiment:", labels[prediction])
    print("Confidence:", round(probabilities[0][prediction].item() * 100, 2), "%")
    unknown = sorted({token for token in tokenize(text) if token not in vocab})
    if unknown:
        print("Unseen tokens mapped to [UNK]:", ", ".join(unknown))

Sentiment: POSITIVE
Confidence: 99.88 %
Unseen tokens mapped to [UNK]: ., and
